# 20. Attention 직접 구현 ★

> **제20장** · **이론편 대응: 17장(Attention), 18.3절(Scaled Dot-Product)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: 없음

---

## 이 장이 특별한 이유

이론편 18.3절에서 **토큰 세 개짜리 Self-Attention을 손으로 끝까지 계산**했다.
이 장에서 그 계산을 **한 단계씩** 코드로 확인한다.

오늘날 거의 모든 언어 모델이 이 계산 위에 서 있다. 여기를 정확히 이해하면
21~22장의 Transformer, 그 뒤의 LLM까지 훨씬 수월해진다.

### 이론편 18.3절에서 손으로 구한 값

| 단계 | 값 |
|---|---|
| 1) 점수 $QK^\top$ 첫 행 | (1, 0, 1) |
| 2) $\sqrt{d_k}$로 나눈 후 | (0.707, 0, 0.707) |
| 3) exp 값 | (2.028, 1.000, 2.028) |
| 4) 합 | 5.056 |
| **5) 가중치 $\alpha_1$** | **(0.401, 0.198, 0.401)** |
| 6) 출력 $\mathbf{z}_1$ | (6.02, 3.98) |

### 진행 방식

**한 셀에 한 단계씩** 다룬다. 각 단계마다 중간값을 전부 출력하고 이론편과 대조한다.
중간에 막히면 그 셀만 다시 실행해 보면 된다.

| 절 | 단계 |
|---|---|
| 1 | Q·K·V가 무엇인가 |
| 2 | **1단계 — 점수 계산** |
| 3 | **2단계 — 스케일링** |
| 4 | **3단계 — 소프트맥스** |
| 5 | **4단계 — 가중합** |
| 6 | 전체를 함수로 |
| 7 | PyTorch와 대조 |
| 8 | Attention 가중치 시각화 |
| 9 | Multi-Head로 확장

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
print(f"NumPy {np.__version__} / PyTorch {torch.__version__}")
print("준비 완료")

---

## 1. Q, K, V가 무엇인가 — 이론편 17.3절

Attention은 세 가지 역할의 벡터를 쓴다. 이론편 17.3절의 도서관 비유를 다시 보자.

| 이름 | 뜻 | 도서관 비유 |
|---|---|---|
| **Q**uery | 지금 무엇을 찾고 있는가 | 내가 찾고 싶은 주제 |
| **K**ey | 각 위치가 어떤 정보를 가졌는가 | 책마다 붙은 분류표 |
| **V**alue | 그 위치의 실제 내용물 | 책의 실제 내용 |

**Self-Attention**은 이 셋을 모두 **같은 문장에서** 만든다.
"이 문장 안의 각 단어가 서로를 얼마나 참고할지"를 스스로 정하는 것이다.

### 이 장에서 쓸 값

이론편 18.3절과 완전히 같은 설정이다. 계산을 손으로 따라갈 수 있도록 아주 작게 잡았다.

$$Q = K = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}, \qquad
V = \begin{pmatrix} 10 & 0 \\ 0 & 10 \\ 5 & 5 \end{pmatrix}$$

각 행이 토큰 하나다. 세 번째 토큰은 앞의 두 방향을 모두 갖고 있어,
앞의 둘과 어느 정도씩 관련이 있는 셈이다.

In [ ]:
import numpy as np

# 이론편 18.3절과 완전히 같은 값
Q = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
K = Q.copy()          # Self-Attention: K도 같은 것에서 나온다
V = np.array([[10.0,  0.0],
              [ 0.0, 10.0],
              [ 5.0,  5.0]])

n_tokens, d_k = Q.shape

print("=" * 55)
print("입력 준비")
print("=" * 55)
print(f"토큰 개수 : {n_tokens}")
print(f"차원 d_k  : {d_k}")
print()
print("Q (Query) — 각 토큰이 무엇을 찾는가")
for i, q in enumerate(Q):
    print(f"  토큰 {i}: {q}")
print()
print("K (Key) — 각 토큰이 무엇을 갖고 있는가")
for i, k in enumerate(K):
    print(f"  토큰 {i}: {k}")
print()
print("V (Value) — 각 토큰의 실제 내용")
for i, v in enumerate(V):
    print(f"  토큰 {i}: {v}")
print()
print("Self-Attention 이므로 Q와 K가 같다.")
print("→ '이 문장 안에서 서로를 얼마나 참고할지'를 계산한다.")

---

## 2. 1단계 — 점수 계산 $QK^\top$

첫 단계는 **모든 토큰 쌍의 유사도**를 재는 것이다. 내적을 쓴다.

이론편 4.5절에서 다뤘듯, 내적이 클수록 두 벡터가 같은 방향을 향한다.
즉 이 값이 "얼마나 관련 있는가"를 나타낸다.

$$S_{ij} = \mathbf{q}_i \cdot \mathbf{k}_j$$

**한 칸씩 손으로 확인해 보자.**

In [ ]:
import numpy as np

print("=" * 60)
print("1단계: 점수 계산 — 한 칸씩")
print("=" * 60)

# 첫 행만 손으로 계산해 본다
print("첫 번째 토큰(Q의 0행)이 각 토큰과 얼마나 관련 있는가")
print(f"  q_0 = {Q[0]}")
print()
for j in range(n_tokens):
    dot = Q[0] @ K[j]
    detail = " + ".join(f"{Q[0][d]:.0f}x{K[j][d]:.0f}" for d in range(d_k))
    print(f"  q_0 · k_{j} = {detail} = {dot:.0f}")
print()

# 전체를 한 번에
scores = Q @ K.T

print("전체 점수 행렬 S = Q @ K.T")
print(scores)
print()
print("이론편 18.3절 값")
print("[[1 0 1]")
print(" [0 1 1]")
print(" [1 1 2]]")
print("-" * 60)

expected_scores = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 2]], dtype=float)
assert np.allclose(scores, expected_scores), "이론편 값과 다릅니다"
print("[OK] 1단계 — 이론편 18.3절과 일치")

### 점수 행렬을 읽는 법

| | 토큰0 | 토큰1 | 토큰2 |
|---|---|---|---|
| **토큰0** | 1 | 0 | 1 |
| **토큰1** | 0 | 1 | 1 |
| **토큰2** | 1 | 1 | 2 |

**행이 "누가 보는가", 열이 "무엇을 보는가"**다.

- 0행 1열이 **0**인 이유: $(1,0)$과 $(0,1)$은 직각이라 내적이 0. **서로 무관**하다는 뜻
- 2행 2열이 **2**로 가장 큰 이유: 자기 자신과의 내적이고 벡터가 길어서

행렬이 대칭인 것은 Self-Attention에서 $Q = K$이기 때문이다.
일반적인 Attention(이론편 17.3절, 번역 등)에서는 대칭이 아니다.

---

## 3. 2단계 — 스케일링 $\div\sqrt{d_k}$

점수를 $\sqrt{d_k}$로 나눈다. 여기서 $d_k = 2$이므로 $\sqrt{2} \approx 1.414$다.

$$\frac{S}{\sqrt{d_k}}$$

**왜 나누는지는 4절 뒤에서 실험으로 확인한다.** 지금은 값만 맞춰 보자.

In [ ]:
import numpy as np

print("=" * 60)
print("2단계: 스케일링")
print("=" * 60)

scale = np.sqrt(d_k)
print(f"d_k = {d_k}")
print(f"sqrt(d_k) = {scale:.6f}")
print()

scaled = scores / scale

print("첫 행 계산")
for j in range(n_tokens):
    print(f"  {scores[0][j]:.0f} / {scale:.4f} = {scaled[0][j]:.6f}")
print()

print("전체 결과")
print(scaled)
print()
print("이론편 18.3절: 첫 행이 (0.707, 0, 0.707)")
print("-" * 60)

assert abs(scaled[0][0] - 0.7071) < 1e-3
assert abs(scaled[0][1] - 0.0) < 1e-9
assert abs(scaled[0][2] - 0.7071) < 1e-3
print("[OK] 2단계 — 이론편 18.3절과 일치")

---

## 4. 3단계 — 소프트맥스 ★

이제 점수를 **합이 1인 가중치**로 바꾼다. 소프트맥스를 쓴다.

$$\alpha_{ij} = \frac{\exp(s_{ij})}{\sum_{j'}\exp(s_{ij'})}$$

**이론편 18.3절에서 첫 행을 손으로 계산했던 과정을 그대로 따라간다.**

1. 각 값에 지수를 취한다
2. 모두 더한다
3. 각각을 합으로 나눈다

In [ ]:
import numpy as np

print("=" * 60)
print("3단계: 소프트맥스 — 첫 행만 손으로")
print("=" * 60)

row = scaled[0]
print(f"입력: {row}")
print()

# ── 3-1) 지수 취하기 ──
print("[3-1] 각 값에 exp 적용")
exp_values = np.exp(row)
for j in range(n_tokens):
    print(f"  exp({row[j]:.6f}) = {exp_values[j]:.6f}")
print(f"  → {exp_values}")
print(f"  이론편: (2.028, 1.000, 2.028)")
print()

# ── 3-2) 합 구하기 ──
total = exp_values.sum()
print("[3-2] 모두 더하기")
print(f"  {exp_values[0]:.6f} + {exp_values[1]:.6f} + {exp_values[2]:.6f}")
print(f"  = {total:.6f}")
print(f"  이론편: 5.056")
print()

# ── 3-3) 나누기 ──
print("[3-3] 각각을 합으로 나누기")
alpha_row = exp_values / total
for j in range(n_tokens):
    print(f"  {exp_values[j]:.6f} / {total:.6f} = {alpha_row[j]:.6f}")
print()
print(f"  결과: {alpha_row}")
print(f"  이론편 : (0.401, 0.198, 0.401)")
print(f"  합  : {alpha_row.sum():.6f}   ← 정확히 1")
print("-" * 60)

assert abs(exp_values[0] - 2.028) < 0.001
assert abs(total - 5.056) < 0.001
assert np.allclose(alpha_row, [0.401, 0.198, 0.401], atol=0.001)
print("[OK] 3단계 — 이론편 18.3절 값과 완전히 일치")

### 이 가중치가 뜻하는 것

$$\alpha_1 = (0.401,\ 0.198,\ 0.401)$$

**첫 번째 토큰이 자기 자신에 40%, 두 번째에 20%, 세 번째에 40%의 주의를 배분한다**는 뜻이다.

두 번째 토큰에 주의를 덜 주는 이유는 2절에서 봤다 — 둘의 내적이 0, 즉 서로 무관한 방향이기 때문이다.

**세 값의 합이 정확히 1**이라는 점이 중요하다. 이 덕분에 "비율"로 해석할 수 있고,
다음 단계에서 가중평균을 낼 수 있다.

In [ ]:
import numpy as np


def softmax(x, axis=-1):
    """수치적으로 안정된 소프트맥스

    최댓값을 빼는 이유:
      exp(1000) 같은 큰 값은 오버플로가 난다.
      모든 값에서 같은 수를 빼도 결과는 같으므로, 최댓값을 빼서 안전하게 만든다.
    """
    x_shifted = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x_shifted)
    return e / e.sum(axis=axis, keepdims=True)


print("=" * 60)
print("최댓값을 빼도 결과가 같은가")
print("=" * 60)
test = np.array([1.0, 2.0, 3.0])
naive = np.exp(test) / np.exp(test).sum()
stable = softmax(test)
print(f"  그냥 계산  : {naive}")
print(f"  최댓값 뺀 것: {stable}")
print(f"  같은가: {np.allclose(naive, stable)}")
print()

print("큰 값에서는 차이가 난다")
big = np.array([1000.0, 1001.0, 1002.0])
try:
    naive_big = np.exp(big) / np.exp(big).sum()
    print(f"  그냥 계산  : {naive_big}   ← nan 이 나온다")
except Exception as e:
    print(f"  그냥 계산  : 오류 {e}")
print(f"  안정된 방식: {softmax(big)}")
print()
print("실무 코드에서 소프트맥스를 이렇게 구현하는 이유다.")

# 전체 가중치 행렬
alpha = softmax(scaled)
print()
print("=" * 60)
print("전체 Attention 가중치")
print("=" * 60)
print(alpha)
print()
print("각 행의 합:", alpha.sum(axis=1))
assert np.allclose(alpha.sum(axis=1), 1.0)
print("[OK] 모든 행의 합이 1")

---

## 5. 4단계 — 가중합 $\alpha V$

마지막이다. 방금 구한 비율대로 **V의 행들을 섞는다.**

$$\mathbf{z}_i = \sum_j \alpha_{ij}\,\mathbf{v}_j$$

이론편 18.3절에서 첫 토큰의 출력을 다음과 같이 계산했다.

$$\mathbf{z}_1 = 0.401 \times (10, 0) + 0.198 \times (0, 10) + 0.401 \times (5, 5) \approx (6.02,\ 3.98)$$

In [ ]:
import numpy as np

print("=" * 60)
print("4단계: 가중합 — 첫 토큰의 출력")
print("=" * 60)

print("각 항의 기여")
contributions = []
for j in range(n_tokens):
    c = alpha[0][j] * V[j]
    contributions.append(c)
    print(f"  {alpha[0][j]:.6f} x {V[j]} = {c}")

print()
print("모두 더하면")
z_0 = sum(contributions)
print(f"  {z_0}")
print(f"  이론편: (6.02, 3.98)")
print()

# 전체를 한 번에
Z = alpha @ V

print("전체 출력 Z = alpha @ V")
print(Z)
print()
print("이론편 18.3절")
print("[[6.02 3.98]")
print(" [3.98 6.02]")
print(" [5.00 5.00]]")
print("-" * 60)

expected_Z = np.array([[6.02, 3.98], [3.98, 6.02], [5.00, 5.00]])
assert np.allclose(Z, expected_Z, atol=0.01), "이론편 값과 다릅니다"
print("[OK] 4단계 — 이론편 18.3절과 일치")

### 결과를 읽는 법

| 토큰 | 원래 V | Attention 출력 |
|---|---|---|
| 0 | (10, 0) | (6.02, 3.98) |
| 1 | (0, 10) | (3.98, 6.02) |
| 2 | (5, 5) | (5.00, 5.00) |

**첫 토큰**의 출력이 원래 값 $(10, 0)$에서 크게 변했다.
자기 정보를 40% 유지하면서 다른 토큰의 정보를 60% 섞어 받은 것이다.

**세 번째 토큰**의 출력이 정확히 $(5, 5)$인 것도 자연스럽다.
이 토큰은 앞의 두 토큰과 똑같은 정도로 관련되어 있으므로 양쪽을 균등하게 반영했다.

이것이 Self-Attention의 본질이다.

> **각 토큰이 문장 안의 다른 토큰들을 얼마나 참고할지 스스로 정하고,
> 그 비율대로 정보를 섞어 자신을 새로 표현한다.**

---

## 6. 전체를 함수로

네 단계를 하나로 묶는다. 이것이 이론편 18.3절의 식 전체다.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
import numpy as np


def scaled_dot_product_attention(Q, K, V, mask=None, verbose=False):
    """Scaled Dot-Product Attention (이론편 18.3절)

    Q: (n_query, d_k)
    K: (n_key,   d_k)
    V: (n_key,   d_v)
    반환: 출력 (n_query, d_v), 가중치 (n_query, n_key)
    """
    d_k = Q.shape[-1]

    # 1단계: 점수
    scores = Q @ K.T

    # 2단계: 스케일링
    scores = scores / np.sqrt(d_k)

    # (선택) 마스킹 — 22장에서 사용
    if mask is not None:
        scores = np.where(mask, scores, -1e9)

    # 3단계: 소프트맥스
    weights = softmax(scores)

    # 4단계: 가중합
    output = weights @ V

    if verbose:
        print(f"  점수      {scores.shape}")
        print(f"  가중치    {weights.shape}  (각 행 합 = 1)")
        print(f"  출력      {output.shape}")

    return output, weights


print("=" * 60)
print("함수로 묶어 다시 확인")
print("=" * 60)
out, w = scaled_dot_product_attention(Q, K, V, verbose=True)
print()
print("출력")
print(out)
print()
print("가중치")
print(w)
print("-" * 60)

assert np.allclose(out, Z), "앞서 단계별로 구한 값과 다릅니다"
assert np.allclose(w, alpha)
print("[OK] 단계별로 구한 값과 일치")

### 왜 $\sqrt{d_k}$로 나누는가 — 실험

3절에서 미뤄 둔 질문에 답할 차례다.

**이유는 확률적이다.** 각 성분이 평균 0, 분산 1인 두 벡터의 내적은
$d_k$개 항의 합이므로 분산이 $d_k$가 되고, 따라서 표준편차는 $\sqrt{d_k}$가 된다.

무작위 벡터로 직접 확인해 보자.

In [ ]:
import numpy as np

print("=" * 60)
print("내적의 표준편차 vs sqrt(d_k)")
print("=" * 60)
print(f"{'d_k':<10}{'내적 표준편차(실측)':<24}{'sqrt(d_k)':<16}{'비율'}")
print("-" * 60)

for d in [2, 8, 64, 512]:
    rng = np.random.RandomState(1)
    q = rng.randn(20000, d)
    k = rng.randn(20000, d)
    dots = (q * k).sum(axis=1)
    std = dots.std()
    print(f"{d:<10}{std:<24.3f}{np.sqrt(d):<16.3f}{std/np.sqrt(d):.3f}")

print("-" * 60)
print("거의 정확히 일치한다.")
print("즉 sqrt(d_k)로 나누면 차원이 얼마든 값의 퍼짐이 1 수준으로 맞춰진다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("스케일링을 하지 않으면 (d_k=64, 점수 격차 10)")
print("=" * 60)

s = np.array([10.0, 0.0, 0.0])

no_scale = softmax(s)
with_scale = softmax(s / 8.0)      # sqrt(64) = 8

print(f"{'':20}{'첫 항':<14}{'둘째':<14}{'셋째'}")
print("-" * 60)
print(f"{'나누지 않으면':20}{no_scale[0]:<14.6f}{no_scale[1]:<14.6f}{no_scale[2]:.6f}")
print(f"{'8로 나누면':20}{with_scale[0]:<14.6f}{with_scale[1]:<14.6f}{with_scale[2]:.6f}")
print("-" * 60)
print()
print("나누지 않으면 첫 항에 99.99%가 몰린다.")
print("나머지 토큰의 정보가 사실상 차단되는 것이다.")
print()
print("더 심각한 문제는 그래디언트다.")
print("소프트맥스 출력이 0이나 1에 붙으면 기울기가 거의 0이 되어 학습이 멈춘다.")
print("→ 이론편 10.3절 시그모이드 포화와 같은 상황")

# 그래디언트가 죽는 것을 확인
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (title, div) in zip(axes, [("스케일링 없음", 1.0), ("sqrt(64)=8로 나눔", 8.0)]):
    gaps = np.linspace(0, 20, 100)
    max_probs = []
    for g in gaps:
        p = softmax(np.array([g, 0.0, 0.0]) / div)
        max_probs.append(p[0])
    ax.plot(gaps, max_probs, linewidth=2.5, color="#EA580C" if div == 1 else "#0D9488")
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("점수 격차")
    ax.set_ylabel("최대 확률")
    ax.set_title(title)
    ax.set_ylim(0.3, 1.05)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("왼쪽: 격차가 조금만 벌어져도 바로 1에 붙는다 (포화)")
print("오른쪽: 완만하게 올라간다 — 그래디언트가 살아 있다")

---

## 7. PyTorch와 대조

직접 만든 것이 PyTorch의 구현과 같은지 확인한다.
PyTorch 2.0부터 `scaled_dot_product_attention` 함수가 내장되어 있다.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

print("=" * 60)
print("직접 구현 vs PyTorch")
print("=" * 60)

# NumPy → PyTorch 텐서
Q_t = torch.tensor(Q, dtype=torch.float32)
K_t = torch.tensor(K, dtype=torch.float32)
V_t = torch.tensor(V, dtype=torch.float32)

# ── 방법 1: 수식 그대로 ──
scores_t = (Q_t @ K_t.T) / np.sqrt(d_k)
weights_t = torch.softmax(scores_t, dim=-1)
out_manual = weights_t @ V_t

# ── 방법 2: PyTorch 내장 함수 ──
out_builtin = F.scaled_dot_product_attention(Q_t, K_t, V_t)

print("직접 구현 (NumPy)")
print(Z)
print()
print("PyTorch 수식 그대로")
print(out_manual.numpy())
print()
print("PyTorch 내장 함수")
print(out_builtin.numpy())
print("-" * 60)

assert np.allclose(Z, out_manual.numpy(), atol=1e-4)
assert np.allclose(Z, out_builtin.numpy(), atol=1e-4)
print("[OK] 세 방법이 모두 일치")
print()
print("내장 함수는 메모리를 아끼는 최적화가 들어 있어 실무에서는 이것을 쓴다.")
print("하지만 안에서 무슨 일이 일어나는지 우리는 이제 안다.")

### 배치 처리

실제로는 여러 문장을 한 번에 처리한다. 텐서 모양이 4차원이 된다.

$$(\text{배치},\ \text{헤드},\ \text{토큰},\ d_k)$$

`@` 연산자는 **마지막 두 차원만** 행렬 곱을 하고 앞쪽은 그대로 두므로,
코드를 거의 바꾸지 않아도 된다.

In [ ]:
import torch
import torch.nn.functional as F

print("=" * 60)
print("배치 처리 시 텐서 모양")
print("=" * 60)

batch, heads, seq, dim = 4, 8, 10, 16
Qb = torch.randn(batch, heads, seq, dim)
Kb = torch.randn(batch, heads, seq, dim)
Vb = torch.randn(batch, heads, seq, dim)

print(f"입력 Q: {tuple(Qb.shape)}   ← (배치, 헤드, 토큰, 차원)")
print()

scores_b = Qb @ Kb.transpose(-2, -1)      # 마지막 두 차원만 전치
print(f"점수   : {tuple(scores_b.shape)}   ← (배치, 헤드, 토큰, 토큰)")

weights_b = torch.softmax(scores_b / dim**0.5, dim=-1)
print(f"가중치 : {tuple(weights_b.shape)}")

out_b = weights_b @ Vb
print(f"출력   : {tuple(out_b.shape)}   ← 입력과 같은 모양")
print()
print(f"가중치 각 행의 합: {weights_b.sum(-1).mean().item():.6f}   (1이어야 함)")
print()
print("주의: transpose(-2, -1) 을 쓴다. .T 는 2차원에서만 안전하다.")
print()

# 내장 함수와 대조
out_builtin_b = F.scaled_dot_product_attention(Qb, Kb, Vb)
print(f"내장 함수와 일치: {torch.allclose(out_b, out_builtin_b, atol=1e-5)}")

---

## 8. Attention 가중치 시각화

Attention의 장점 중 하나는 **모델이 무엇을 보고 있는지 들여다볼 수 있다**는 것이다.
가중치 행렬을 그림으로 그려 보자.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- 왼쪽: 우리가 계산한 가중치 ---
ax = axes[0]
im = ax.imshow(alpha, cmap="Blues", vmin=0, vmax=0.6)
plt.colorbar(im, ax=ax, label="주의 비율")
for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f"{alpha[i,j]:.3f}", ha="center", va="center",
                fontsize=11, color="white" if alpha[i,j] > 0.35 else "black")
ax.set_xticks(range(n_tokens)); ax.set_xticklabels([f"토큰{j}" for j in range(n_tokens)])
ax.set_yticks(range(n_tokens)); ax.set_yticklabels([f"토큰{i}" for i in range(n_tokens)])
ax.set_xlabel("보는 대상")
ax.set_ylabel("보는 주체")
ax.set_title("Attention 가중치")

# --- 오른쪽: 첫 토큰의 배분 ---
ax = axes[1]
colors = ["#1E40AF", "#94A3B8", "#0D9488"]
bars = ax.bar(range(n_tokens), alpha[0], color=colors)
for i, (b, v) in enumerate(zip(bars, alpha[0])):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}",
            ha="center", fontsize=10)
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(["토큰0\n(자기)", "토큰1\n(무관)", "토큰2\n(관련)"])
ax.set_ylabel("주의 비율")
ax.set_title("토큰 0이 주의를 배분한 방식")
ax.set_ylim(0, 0.5)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("왼쪽 그림의 대각선이 밝다 = 자기 자신을 많이 참고한다")
print("토큰0과 토큰1이 서로 어둡다 = 내적이 0이라 무관하다고 판단")
print()
print("실제 언어 모델에서 이런 그림을 그리면")
print("대명사가 어느 명사를 가리키는지 같은 것이 드러나기도 한다.")

### 실제 문장으로 해보기

의미를 가진 예로 확인한다. "나는 학교에 간다"를 번역하는 상황을 흉내 낸다.
각 단어를 간단한 벡터로 표현하고, 어느 단어가 어느 단어를 참고하는지 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 단어를 벡터로 (실제로는 24장의 Embedding이 이 역할)
words = ["나는", "학교에", "간다", "매일"]
# 의도적으로 구성: '나는'과 '간다'가 관련(주어-동사), '학교에'와 '간다'도 관련
E = np.array([
    [1.0, 0.2, 0.0],    # 나는   — 주어 성격
    [0.1, 1.0, 0.3],    # 학교에 — 장소 성격
    [0.8, 0.7, 0.2],    # 간다   — 주어·장소 모두와 관련
    [0.0, 0.2, 1.0],    # 매일   — 시간 성격
])

out_w, attn_w = scaled_dot_product_attention(E, E, E)

print("=" * 60)
print("문장에 Self-Attention 적용")
print("=" * 60)
print(f"{'':10}" + "".join(f"{w:>10}" for w in words))
print("-" * 60)
for i, w in enumerate(words):
    print(f"{w:<10}" + "".join(f"{attn_w[i,j]:>10.3f}" for j in range(len(words))))
print("-" * 60)
print()

for i, w in enumerate(words):
    top = np.argsort(attn_w[i])[::-1][:2]
    print(f"  '{w}'가 가장 많이 본 것: "
          f"'{words[top[0]]}'({attn_w[i,top[0]]:.3f}), "
          f"'{words[top[1]]}'({attn_w[i,top[1]]:.3f})")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(attn_w, cmap="Blues")
plt.colorbar(im, ax=ax)
for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f"{attn_w[i,j]:.2f}", ha="center", va="center",
                fontsize=10, color="white" if attn_w[i,j] > 0.3 else "black")
ax.set_xticks(range(len(words))); ax.set_xticklabels(words)
ax.set_yticks(range(len(words))); ax.set_yticklabels(words)
ax.set_xlabel("보는 대상")
ax.set_ylabel("보는 주체")
ax.set_title("문장 내 Attention")
plt.tight_layout()
plt.show()

print()
print("'간다'가 '나는'과 '학교에'를 함께 참고하는 것이 보인다.")
print("벡터를 그렇게 설계했으니 당연하지만, 실제 모델은 이것을 학습으로 찾아낸다.")

---

## 9. Multi-Head로의 확장 — 이론편 18.4절

지금까지는 Attention을 **한 번만** 계산했다. 실제 Transformer는 여러 번 병렬로 한다.

**왜 그럴까.** 이론편 18.4절에서 다룬 이유는 이렇다.

문장을 이해할 때 우리는 여러 층위의 관계를 동시에 본다.
- 어떤 단어가 문법적으로 어느 단어에 걸리는지
- 의미상 무엇을 가리키는지
- 어떤 단어와 짝을 이루는 관용 표현인지

**Attention 하나로는 한 종류의 관계밖에 못 담는다.** 그래서 여러 "머리"를 둔다.

각 머리는 전체 차원을 나눠 갖는다. $d_{model} = 8$, 헤드 4개면 각 머리는 2차원씩이다.

In [ ]:
import numpy as np

print("=" * 60)
print("Multi-Head 개념 확인")
print("=" * 60)

d_model = 8
n_heads = 4
d_head = d_model // n_heads

print(f"전체 차원 d_model : {d_model}")
print(f"헤드 개수         : {n_heads}")
print(f"헤드당 차원       : {d_head}   ({d_model} / {n_heads})")
print()
print("전체 차원을 나눠 갖는다 — 계산량은 한 번 할 때와 비슷하다.")
print()

# 간단한 예: 토큰 5개
rng = np.random.RandomState(0)
n_tok = 5
X = rng.randn(n_tok, d_model)

print(f"입력 X: {X.shape}")
print()

# 헤드별로 나눠서 각각 Attention
head_outputs = []
head_weights = []
for h in range(n_heads):
    start, end = h * d_head, (h + 1) * d_head
    Qh = X[:, start:end]
    Kh = X[:, start:end]
    Vh = X[:, start:end]
    out_h, w_h = scaled_dot_product_attention(Qh, Kh, Vh)
    head_outputs.append(out_h)
    head_weights.append(w_h)
    print(f"  헤드 {h}: 입력 {Qh.shape} → 출력 {out_h.shape}")

# 다시 이어 붙인다
concat = np.concatenate(head_outputs, axis=-1)
print()
print(f"이어 붙인 결과: {concat.shape}   ← 입력과 같은 모양")
assert concat.shape == X.shape
print("[OK] 모양이 유지된다 — 층을 쌓을 수 있다")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 각 헤드가 서로 다른 패턴을 보는지 확인
fig, axes = plt.subplots(1, n_heads, figsize=(14, 3.2))

for h, ax in enumerate(axes):
    im = ax.imshow(head_weights[h], cmap="Blues", vmin=0, vmax=0.6)
    ax.set_title(f"헤드 {h}", fontsize=10)
    ax.set_xticks(range(n_tok))
    ax.set_yticks(range(n_tok))
    ax.tick_params(labelsize=7)

fig.suptitle("헤드마다 다른 곳을 본다", fontsize=12)
plt.tight_layout()
plt.show()

# 헤드 간 차이를 수치로
print("=" * 55)
print("헤드 간 가중치 차이")
print("=" * 55)
from itertools import combinations
diffs = []
for h1, h2 in combinations(range(n_heads), 2):
    d = np.abs(head_weights[h1] - head_weights[h2]).mean()
    diffs.append(d)
    print(f"  헤드 {h1} vs 헤드 {h2}: 평균 차이 {d:.4f}")
print("-" * 55)
print(f"평균: {np.mean(diffs):.4f}")
print()
print("헤드마다 다른 부분 공간을 보므로 서로 다른 패턴이 나온다.")
print()
print("여기서는 X를 그냥 잘라 썼지만, 실제 Transformer는")
print("각 헤드마다 별도의 W_Q, W_K, W_V 를 학습한다.")
print("→ 21장에서 제대로 구현한다.")

---

## 10. 정리

### 확인한 이론편 값 (18.3절 전 과정)

| 단계 | 코드 결과 | 이론편 |
|---|---|---|
| 1) 점수 첫 행 | (1, 0, 1) | 일치 ✓ |
| 2) 스케일링 후 | (0.707, 0, 0.707) | 일치 ✓ |
| 3) exp 값 | (2.028, 1.000, 2.028) | 일치 ✓ |
| 4) 합 | 5.056 | 일치 ✓ |
| **5) 가중치** | **(0.401, 0.198, 0.401)** | **일치** ✓ |
| 6) 출력 | (6.02, 3.98) | 일치 ✓ |

**네 단계 전부 손계산과 맞았다.** 여기에 스케일링의 근거(내적 표준편차 = √d_k)도 실험으로 확인했다.

### Attention 네 단계 요약

```python
scores  = Q @ K.T           # 1) 모든 쌍의 유사도
scores /= np.sqrt(d_k)      # 2) 크기 조정
weights = softmax(scores)   # 3) 합이 1인 비율로
output  = weights @ V       # 4) 그 비율대로 섞기
```

**네 줄이 전부다.** 이 위에 오늘날의 언어 모델이 서 있다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| Q·K·V | 찾는 것 / 가진 것 / 실제 내용 |
| 점수 행렬 | 행=보는 주체, 열=보는 대상 |
| √d_k | 내적의 표준편차가 √d_k이므로 나눠서 1 수준으로 |
| 소프트맥스 | 최댓값을 빼서 오버플로 방지 |
| 가중합 | 각 토큰이 정보를 섞어 자신을 새로 표현 |
| 배치 처리 | `transpose(-2,-1)` 사용 (`.T`는 2차원 전용) |
| Multi-Head | 여러 관계를 동시에 — 차원을 나눠 가짐 |

### RNN과 무엇이 다른가 (16장과 비교)

| | RNN/LSTM | Attention |
|---|---|---|
| 계산 순서 | 순차적 (t-1 → t) | **모든 쌍 동시에** |
| 먼 토큰 접근 | 여러 단계 거쳐야 | **한 번에** |
| 정보 병목 | 고정 크기 벡터로 압축 | **압축 없음** |
| 병렬화 | 불가 | 가능 |

16장 7절에서 본 두 한계가 모두 해결된 것이다.

### 다음 장

**21. Transformer Encoder 직접 구현** — Attention 위에 나머지 부품을 얹는다.
Positional Encoding, Multi-Head, Residual, LayerNorm을 붙여 인코더 블록을 완성한다.